In [1]:
import json
import random

random.seed(42)

INPUT_FILE = "gov_data_uk.json"
OUTPUT_FILE = "dataset_sample_10k.json"

TOTAL_SAMPLE = 10000

# Filtering the metadata in english
def is_english(text):
    try:
        text.encode("ascii")
    except UnicodeEncodeError:
        return False

    # simple heuristic: contains common English words
    common_words = ["the", "and", "of", "in", "to"]
    text_lower = text.lower()
    return any(word in text_lower for word in common_words)

# LOADIND DATASETS
with open(INPUT_FILE, "r", encoding="utf-8") as f:
    data = json.load(f)

print(f"Loaded {len(data)} datasets")

# CLEANING
seen_ids = set()

clean_data = []
duplicates = 0
incomplete = 0
non_english = 0

for d in data:
    dataset_id = d.get("dataset_id")
    title = d.get("title")
    description = d.get("description")
    keywords = d.get("keywords")

    # completeness check 
    if (
        not dataset_id or
        not title or
        not description or
        not keywords or
        len(keywords) == 0
    ):
        incomplete += 1
        continue

    #  english check 
    if not is_english(title + " " + description):
        non_english += 1
        continue

    #  deduplication (ONLY dataset_id)
    if dataset_id in seen_ids:
        duplicates += 1
        continue

    seen_ids.add(dataset_id)
    clean_data.append(d)

print(f"Removed {duplicates} duplicate dataset_ids")
print(f"Removed {incomplete} incomplete datasets")
print(f"Removed {non_english} non-English datasets")
print(f"Remaining clean datasets: {len(clean_data)}")

#  RANDOM SAMPLING
if len(clean_data) < TOTAL_SAMPLE:
    raise ValueError(f"Not enough clean datasets to sample {TOTAL_SAMPLE}")

sampled_data = random.sample(clean_data, TOTAL_SAMPLE)

print(f"Randomly selected {len(sampled_data)} datasets")

# SAVE 
with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
    json.dump(sampled_data, f, indent=2, ensure_ascii=False)

# A REPORT
print("\n--- FINAL DATASET ---")
print(f"Total datasets: {len(sampled_data)}")

Loaded 23777 datasets
Removed 0 duplicate dataset_ids
Removed 74 incomplete datasets
Removed 12310 non-English datasets
Remaining clean datasets: 11393
Randomly selected 10000 datasets

--- FINAL DATASET ---
Total datasets: 10000
